# Analisi dei run Max-Cut

Il notebook legge i file `result.json` e `graph.json` prodotti dagli esperimenti e mostra metriche di qualità, tempo e consumo energetico fisico quando misurato.

Per scegliere i dati, modifica `RUNS_FOLDER` nella prima cella di codice:
- `None`: analizza tutti i run trovati sotto `results/`;
- una cartella di esperimento, la sua sottocartella `runs/` oppure una singola cartella `run_*`: analizza solo i run contenuti lì.

I percorsi relativi sono risolti dalla radice del repository. `cut_value` è il taglio ricalcolato dagli archi e dalla soluzione; `energy = -cut_value` è l'energia matematica dell'obiettivo Max-Cut, non il consumo fisico. Quest'ultimo è riportato separatamente come `physical_energy_joules` solo per le misure con stato `measured`; unità, metodo, ambito e motivo di indisponibilità sono mostrati per run. `cut_fraction` normalizza il taglio rispetto alla somma dei valori assoluti dei pesi (utile anche per i pesi GSET negativi), mentre `best_observed_ratio` confronta il taglio con il miglior risultato caricato per lo stesso grafo. Quest'ultimo è un confronto empirico, non un rapporto rispetto all'ottimo noto.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
RESULTS_ROOT = REPO_ROOT / "results"

# None analizza tutti i risultati; altrimenti imposta un percorso relativo alla radice del repository
# o un percorso assoluto, per esempio Path(r"results\20261019_214618_benchmark_cpu\runs").
RUNS_FOLDER = None

selected_folder = RESULTS_ROOT if RUNS_FOLDER is None else Path(RUNS_FOLDER)
if not selected_folder.is_absolute():
    selected_folder = REPO_ROOT / selected_folder
selected_folder = selected_folder.resolve()

if not selected_folder.is_dir():
    raise FileNotFoundError(f"Cartella dei run non trovata: {selected_folder}")

result_files = (
    [selected_folder / "result.json"]
    if (selected_folder / "result.json").is_file()
    else sorted(selected_folder.rglob("result.json"))
)
print(f"Cartella analizzata: {selected_folder}")
print(f"File result.json trovati: {len(result_files)}")

## Caricamento e ricalcolo delle metriche

Per ogni run il valore del taglio viene ricalcolato usando gli archi in `graph.json` e gli spin in `result.json` (o, in alternativa, `solution.json`). File mancanti o non validi vengono elencati nella colonna degli errori, senza interrompere l'analisi degli altri run.

In [ ]:
rows = []
errors = []

for result_file in result_files:
    run_dir = result_file.parent
    try:
        result = json.loads(result_file.read_text(encoding="utf-8"))
        graph_file = run_dir / "graph.json"
        if not graph_file.is_file():
            raise FileNotFoundError("graph.json mancante")
        graph = json.loads(graph_file.read_text(encoding="utf-8"))

        spins = result.get("solution")
        if spins is None:
            solution_file = run_dir / "solution.json"
            if not solution_file.is_file():
                raise FileNotFoundError("soluzione assente sia in result.json sia in solution.json")
            spins = json.loads(solution_file.read_text(encoding="utf-8"))["solution_spins"]
        if len(spins) != graph["n_nodes"] or any(spin not in (-1, 1) for spin in spins):
            raise ValueError("la soluzione deve contenere uno spin -1 o +1 per ogni nodo")

        edges = graph["edges"]
        total_edge_weight = sum(abs(float(edge["weight"])) for edge in edges)
        cut_value = sum(
            float(edge["weight"])
            for edge in edges
            if spins[int(edge["u"])] != spins[int(edge["v"])]
        )
        physical_energy = result.get("physical_energy") or {}
        if not isinstance(physical_energy, dict):
            physical_energy = {}
        energy_status = physical_energy.get("status", "unavailable")
        energy_joules = (
            physical_energy.get("joules")
            if energy_status == "measured"
            else None
        )
        rows.append({
            "run_folder": str(run_dir),
            "run_id": result.get("run_id", run_dir.name),
            "graph_id": result.get("graph_id", graph.get("name", "sconosciuto")),
            "graph_type": result.get("graph_type", graph.get("type", "other")),
            "n_nodes": result.get("n_nodes", graph["n_nodes"]),
            "density": result.get("density"),
            "algorithm": result.get("algorithm", "sconosciuto"),
            "backend": result.get("backend", result.get("hardware", "sconosciuto")),
            "status": result.get("status", "sconosciuto"),
            "reported_objective": result.get("objective_value"),
            "cut_value": cut_value,
            "energy": -cut_value,
            "total_edge_weight": total_edge_weight,
            "cut_fraction": cut_value / total_edge_weight if total_edge_weight > 0 else float("nan"),
            "runtime_seconds": result.get("runtime_seconds"),
            "physical_energy_joules": energy_joules,
            "physical_energy_unit": physical_energy.get("unit", "J"),
            "physical_energy_status": energy_status,
            "physical_energy_method": physical_energy.get("method"),
            "physical_energy_scope": physical_energy.get("scope"),
            "physical_energy_error": physical_energy.get("error"),
        })
    except (OSError, json.JSONDecodeError, KeyError, TypeError, ValueError) as exc:
        errors.append({"run_folder": str(run_dir), "error": f"{type(exc).__name__}: {exc}"})

if not rows:
    print("Nessun run valido trovato. Verifica RUNS_FOLDER e che i risultati contengano result.json e graph.json.")

runs = pd.DataFrame(rows)
if not runs.empty:
    best_by_graph = runs.groupby("graph_id")["cut_value"].transform("max")
    runs["best_observed_ratio"] = runs["cut_value"].div(best_by_graph.where(best_by_graph > 0))

if errors:
    print(f"\nRun saltati per dati mancanti o non validi ({len(errors)}):")
    display(pd.DataFrame(errors))
elif result_files:
    print("Tutti i run trovati sono stati caricati correttamente.")

## Risultati per run

In [ ]:
if not runs.empty:
    display(runs.sort_values(["graph_id", "algorithm", "run_id"]).reset_index(drop=True))

## Riepilogo per grafo e algoritmo

Tempi, qualità e consumo energetico fisico misurato sono aggregati separatamente per grafo, algoritmo e backend. Il numero di misure disponibili accompagna la media dei joule; run senza misura non vengono interpretati come consumo zero.

In [ ]:
if not runs.empty:
    summary = (
        runs.groupby(["graph_id", "algorithm", "backend"], dropna=False)
        .agg(
            run_count=("run_id", "count"),
            cut_value_mean=("cut_value", "mean"),
            cut_value_best=("cut_value", "max"),
            energy_mean=("energy", "mean"),
            cut_fraction_mean=("cut_fraction", "mean"),
            best_observed_ratio_mean=("best_observed_ratio", "mean"),
            runtime_mean_seconds=("runtime_seconds", "mean"),
            runtime_median_seconds=("runtime_seconds", "median"),
            physical_energy_joules_mean=("physical_energy_joules", "mean"),
            physical_energy_measurement_count=("physical_energy_joules", "count"),
        )
        .reset_index()
    )
    display(summary)

## Grafici

I grafici confrontano la distribuzione dei tempi, la frazione del peso tagliata e, quando disponibile, il consumo fisico misurato in joule per backend.

In [ ]:
if not runs.empty:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    runs.boxplot(column="runtime_seconds", by="algorithm", ax=axes[0], rot=30)
    axes[0].set_title("Tempo di esecuzione per algoritmo")
    axes[0].set_xlabel("Algoritmo")
    axes[0].set_ylabel("Secondi")

    quality = runs.pivot_table(
        index="graph_id", columns="algorithm", values="cut_fraction", aggfunc="mean"
    )
    quality.plot(kind="bar", ax=axes[1])
    axes[1].set_title("Frazione del peso degli archi tagliata")
    axes[1].set_xlabel("Grafo")
    axes[1].set_ylabel("Peso tagliato / peso totale")
    axes[1].set_ylim(0, 1)
    axes[1].legend(title="Algoritmo")

    measured_energy = runs.dropna(subset=["physical_energy_joules"])
    if measured_energy.empty:
        axes[2].text(0.5, 0.5, "Nessuna misura energetica disponibile", ha="center", va="center")
        axes[2].set_axis_off()
    else:
        measured_energy.boxplot(column="physical_energy_joules", by="backend", ax=axes[2], rot=30)
        axes[2].set_title("Consumo fisico per backend")
        axes[2].set_xlabel("Backend")
        axes[2].set_ylabel("Energia misurata (J)")

    fig.tight_layout()
    plt.show()
    plt.close(fig)

## Trade-off: tempo, qualità, dimensione ed energia

Le figure seguenti usano solo run completate per qualità e tempo. I grafici energetici e la frontiera di Pareto includono esclusivamente misure fisiche con stato `measured`; le misure mancanti non vengono imputate. Per ER la famiglia è classificata da densità (<0.1 sparso, >=0.5 denso, altrimenti media); GSET e gli altri tipi restano categorie distinte. Il baseline qualità usa Exact CPU quando presente sul grafo, altrimenti il miglior risultato osservato. Lo speedup usa la run CPU mediana dello stesso grafo e algoritmo.

Impostare `FIGURES_FOLDER` per esportare le figure in PNG e PDF.


In [ ]:
from analysis.plots import (
    plot_time_quality,
    plot_measured_energy_tradeoffs,
    plot_quality_by_graph_size,
    plot_scaling,
    plot_normalized_comparison,
    plot_pareto_frontier,
)

# Percorso opzionale; None mantiene le figure soltanto nel notebook.
FIGURES_FOLDER = None  # esempio: REPO_ROOT / "results" / "figures"

if runs.empty:
    print("Nessun run valido: le figure non possono essere generate.")
else:
    plot_time_quality(runs, output_dir=FIGURES_FOLDER)
    plot_measured_energy_tradeoffs(runs, output_dir=FIGURES_FOLDER)
    plot_quality_by_graph_size(runs, output_dir=FIGURES_FOLDER)
    plot_scaling(runs, output_dir=FIGURES_FOLDER)
    normalized_runs, normalized_figure = plot_normalized_comparison(
        runs, baseline="exact", output_dir=FIGURES_FOLDER
    )
    display(normalized_runs[[
        "run_id", "graph_id", "algorithm", "backend", "quality_ratio",
        "quality_reference", "speedup_vs_cpu"
    ]].sort_values(["graph_id", "algorithm", "backend"]))
    pareto_runs, pareto_figure = plot_pareto_frontier(
        runs, output_dir=FIGURES_FOLDER
    )
    display(pareto_runs.loc[pareto_runs["pareto_optimal"], [
        "run_id", "graph_id", "algorithm", "backend", "runtime_seconds",
        "measured_energy_joules", "cut_fraction"
    ]].sort_values(["graph_id", "runtime_seconds"]))
